In [1]:
import os, sys, torch
os.chdir('/Users/neiltripathi/mech-interp')
sys.path.insert(0, '/Users/neiltripathi/mech-interp/scripts')
print(os.getcwd())
print(open('scripts/transformer_bpe.py').read())

/Users/neiltripathi/mech-interp
import os
import time
import torch
import torch.nn as nn
from torch.nn import functional as F # stateless functions from nn, no stored params
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# repo root = parent of scripts/, so data + checkpoint paths resolve no matter the cwd
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

# ── induction_2 ───────────────────────────────────────────────────────────────
# This is transformer.py with ONE variable changed: a trained byte-level BPE
# tokenizer in place of the 65-character vocabulary. Architecture, training loop,
# hyperparameters, and the checkpoint schedule are identical — a controlled test
# of whether subword tokens give induction heads a key worth matching on (char
# keys don't: every "P" is identical, so the match step has nothing to grab).
# Artifacts go to models/bpe/ so they never clobber the char model in models/.
# ──────────────────────────────────

In [2]:
import os, sys, torch
os.chdir('/Users/neiltripathi/mech-interp')
sys.path.insert(0, '/Users/neiltripathi/mech-interp/scripts')
from transformer_bpe import GPTLanguageModel, encode, decode, vocab_size

print('vocab_size:', vocab_size)   # expect 4096

vocab_size: 4096


In [3]:
model = GPTLanguageModel()
model.load_state_dict(torch.load('models/bpe/ckpt_001000.pt'))
model.eval()
model.to('mps')

# verify with a forward pass on val text
with open('input.txt') as f:
    text = f.read()
ids = encode(text[-5000:])[:257]      # note: encode first, THEN slice — BPE tokens, not chars
x = torch.tensor(ids[:-1]).unsqueeze(0).to('mps')
y = torch.tensor(ids[1:]).unsqueeze(0).to('mps')
with torch.no_grad():
    _, loss = model(x, y)
print('loss:', loss.item())   # expect ~4.5

loss: 4.972288608551025


In [4]:
def make_hook(layer, head):
    def hook(module, inp, out):
        cache[(layer, head)] = out.detach()
    return hook

seq_len = 50
rand = torch.randint(0, vocab_size, (seq_len,))   # NOW from 4096, not 65
rep = torch.cat([rand, rand]).unsqueeze(0).to('mps')

def score_current_model():
    global cache
    cache = {}
    handles = [model.blocks[L].sa.heads[H].dropout.register_forward_hook(make_hook(L, H))
               for L in range(6) for H in range(6)]
    with torch.no_grad():
        model(rep)
    for h in handles:
        h.remove()
    prev_best, ind_best = 0.0, 0.0
    for L in range(6):
        for H in range(6):
            wei = cache[(L, H)][0]
            prev = sum(wei[i, i-1].item() for i in range(1, 2*seq_len)) / (2*seq_len - 1)
            ind  = sum(wei[i, i-seq_len+1].item() for i in range(seq_len, 2*seq_len)) / seq_len
            prev_best = max(prev_best, prev)
            ind_best  = max(ind_best, ind)
    return prev_best, ind_best

print(score_current_model())

(0.3835343840447339, 0.013209253118839115)


In [5]:
import glob
ckpts = sorted(glob.glob('models/bpe/ckpt_*.pt'))
steps, prev_scores, ind_scores = [], [], []
for path in ckpts:
    step = int(path.split('_')[-1].split('.')[0])
    model.load_state_dict(torch.load(path))
    model.eval()
    p, i = score_current_model()
    steps.append(step); prev_scores.append(p); ind_scores.append(i)
    print(f"step {step:4d}:  prev-token {p:.3f}   induction {i:.3f}")

step    0:  prev-token 0.048   induction 0.016
step   10:  prev-token 0.047   induction 0.015
step   25:  prev-token 0.048   induction 0.015
step   50:  prev-token 0.060   induction 0.015
step  100:  prev-token 0.051   induction 0.016
step  150:  prev-token 0.052   induction 0.017
step  200:  prev-token 0.056   induction 0.015
step  250:  prev-token 0.066   induction 0.015
step  350:  prev-token 0.089   induction 0.015
step  500:  prev-token 0.148   induction 0.014
step  750:  prev-token 0.259   induction 0.014
step 1000:  prev-token 0.384   induction 0.013
step 1250:  prev-token 0.511   induction 0.013
step 1500:  prev-token 0.601   induction 0.013
step 1750:  prev-token 0.670   induction 0.014
step 2000:  prev-token 0.721   induction 0.013
step 2250:  prev-token 0.748   induction 0.013
step 2500:  prev-token 0.782   induction 0.013
step 2750:  prev-token 0.811   induction 0.013
step 3000:  prev-token 0.826   induction 0.013
step 3250:  prev-token 0.837   induction 0.013
step 3500:  p

In [6]:
# real repeated text probe: a Shakespeare passage, then the SAME passage again
passage = text[1000:1000+400]          # 400 chars of real Shakespeare
ids = encode(passage)                   # BPE tokens
half = len(ids)
rep_real = torch.tensor(ids + ids).unsqueeze(0).to('mps')   # passage twice
print('tokens per copy:', half, '| total:', 2*half)

# make sure it fits in block_size (256). if 2*half > 256, shorten the passage.
assert 2*half <= 256, f"too long ({2*half}); cut the passage"

# load the trained model (step 1000, best val) for this test
model.load_state_dict(torch.load('models/bpe/ckpt_001000.pt'))
model.eval()

cache = {}
handles = [model.blocks[L].sa.heads[H].dropout.register_forward_hook(make_hook(L, H))
           for L in range(6) for H in range(6)]
with torch.no_grad():
    model(rep_real)
for h in handles:
    h.remove()

# induction score: in the SECOND copy, does position attend `half` back +1?
ind_best, ind_where = 0.0, None
for L in range(6):
    for H in range(6):
        wei = cache[(L, H)][0]
        total = sum(wei[i, i-half+1].item() for i in range(half, 2*half)) / half
        if total > ind_best:
            ind_best, ind_where = total, (L, H)
print(f'best induction on REAL repeated text: {ind_best:.3f} at head {ind_where}')

tokens per copy: 113 | total: 226
best induction on REAL repeated text: 0.006 at head (0, 0)
